In [ ]:
import scanpy as sc
import numpy as np
import flowkit as fk
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from preprocessing import read_flow
bom_dir = 'BOM_CD3_01DEC25'
bom_flow, bom_samples, bom_session = read_flow(bom_dir, "BOM")

In [ ]:
lln_dir = 'LLN_CD3_01DEC25'
lln_flow, lln_samples, lln_session = read_flow(lln_dir, 'LLN')

In [ ]:
lng_dir = 'LNG_CD3_01DEC25'
lng_flow, lng_samples, lng_session = read_flow(lng_dir, 'LNG')

In [ ]:
mln_dir = 'MLN_CD3_01DEC25'
mln_flow, mln_samples, mln_session = read_flow(mln_dir, 'MLN')

In [ ]:
spl_dir = 'SPL_CD3_01DEC25'
spl_flow, spl_samples, spl_session = read_flow(spl_dir, 'SPL')

In [ ]:
df_flow = pd.concat([bom_flow, lln_flow, lng_flow, mln_flow, spl_flow])

In [ ]:
for var in ['bom_flow', 'lln_flow', 'lng_flow', 'mln_flow', 'spl_flow']:
    del globals()[var]

In [ ]:
df_flow['PD1'] = df_flow[['PD1', 'PD-1']].max(axis=1)
df_flow['TCRva'] = df_flow[['TCRva', 'TCRVaJa']].max(axis=1)
df_flow['41BB'] = df_flow[['41BB', '4-1BB']].max(axis=1)
df_flow['CD103'] = df_flow[['CD103', 'CD193']].max(axis=1)

In [ ]:
df_flow.drop(columns = ['PD-1', 'TCRVaJa', '4-1BB', 'CD193'], inplace=True)

In [ ]:
exclude = ['FSC-A', 'FSC-H', 'SSC-A', 'SSC-B-A', 'SSC-B-H', 'SSC-H', 'AF-A', 'CD66bCD19CD326LD', 'Time', 'CD45', 'Event #']
df_flow = df_flow.drop(columns=exclude)

In [ ]:
df_flow_age_low = df_flow[df_flow.age < 40]
df_flow_age_high = df_flow[df_flow.age > 40]

In [ ]:
df_flow_counts_low = df_flow_age_low[
    df_flow_age_low.select_dtypes(include=[np.number]).columns.difference(["age"])
]

In [ ]:
df_flow_counts_high = df_flow_age_high[
    df_flow_age_high.select_dtypes(include=[np.number]).columns.difference(["age"])
]

In [ ]:
from preprocessing import pd_to_adata
adata_low = pd_to_adata(df_flow_age_low, df_flow_counts_low)
adata_high = pd_to_adata(df_flow_age_high, df_flow_counts_high)

In [ ]:
adata_low.X = np.arcsinh(adata_low.X / 150)
adata_high.X = np.arcsinh(adata_high.X / 150)

In [ ]:
sc.pp.scale(adata_low, max_value=5)
sc.pp.scale(adata_high, max_value=5)

In [ ]:
adata_low = adata_low[(adata_low[:, 'CD3'].X > 0)]
adata_low = adata_low[:, adata_low.var.index != 'CD3']

adata_high = adata_high[(adata_high[:, 'CD3'].X > 0)]
adata_high = adata_high[:, adata_high.var.index != 'CD3']

print(adata_low, adata_high)

In [ ]:
CD4_low = adata_low[(adata_low[:, 'CD4'].X > 0)]
CD4_low = CD4_low[:, CD4_low.var.index != 'gdTCR']
CD4_low = CD4_low[:, CD4_low.var.index != 'TCRva']
CD4_low = CD4_low[:, CD4_low.var.index != 'CD4']
CD4_low = CD4_low[:, CD4_low.var.index != 'CD8']

groups = ['ctr', 'hst', 'ftl']
tissues = CD4_low.obs['tissue'].unique().tolist()
adata_list = []

for group in groups:
    for tissue in tissues:
        subset = CD4_low[(CD4_low.obs['group'] == group) & (CD4_low.obs['tissue'] == tissue)]
        if subset.shape[0] >= 5000:
            sampled_subset = subset[np.random.choice(subset.shape[0], 5000, replace=False)]
        else:
            sampled_subset = subset  
        
        sampled_subset.obs['group'] = f"{group}"
        adata_list.append(sampled_subset.copy())

CD4_low = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
CD8_low = adata_low[(adata_low[:, 'CD8'].X > 0)]
CD8_low = CD8_low[:, CD8_low.var.index != 'gdTCR']
CD8_low = CD8_low[:, CD8_low.var.index != 'TCRva']
CD8_low = CD8_low[:, CD8_low.var.index != 'CD4']
CD8_low = CD8_low[:, CD8_low.var.index != 'CD8']

groups = ['ctr', 'hst', 'ftl']
tissues = CD8_low.obs['tissue'].unique().tolist()
adata_list = []

for group in groups:
    for tissue in tissues:
        subset = CD8_low[(CD8_low.obs['group'] == group) & (CD8_low.obs['tissue'] == tissue)]
        if subset.shape[0] >= 5000:
            sampled_subset = subset[np.random.choice(subset.shape[0], 5000, replace=False)]
        else:
            sampled_subset = subset  
        
        sampled_subset.obs['group'] = f"{group}"
        adata_list.append(sampled_subset.copy())

CD8_low = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
CD4_high = adata_high[(adata_high[:, 'CD4'].X > 0)]
CD4_high = CD4_high[:, CD4_high.var.index != 'gdTCR']
CD4_high = CD4_high[:, CD4_high.var.index != 'TCRva']
CD4_high = CD4_high[:, CD4_high.var.index != 'CD4']
CD4_high = CD4_high[:, CD4_high.var.index != 'CD8']

groups = ['ctr', 'hst', 'ftl']
tissues = CD4_high.obs['tissue'].unique().tolist()
adata_list = []

for group in groups:
    for tissue in tissues:
        subset = CD4_high[(CD4_high.obs['group'] == group) & (CD4_high.obs['tissue'] == tissue)]
        if subset.shape[0] >= 5000:
            sampled_subset = subset[np.random.choice(subset.shape[0], 5000, replace=False)]
        else:
            sampled_subset = subset  
        
        sampled_subset.obs['group'] = f"{group}"
        adata_list.append(sampled_subset.copy())

CD4_high = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
CD8_high = adata_high[(adata_high[:, 'CD8'].X > 0)]
CD8_high = CD8_high[:, CD8_high.var.index != 'gdTCR']
CD8_high = CD8_high[:, CD8_high.var.index != 'TCRva']
CD8_high = CD8_high[:, CD8_high.var.index != 'CD4']
CD8_high = CD8_high[:, CD8_high.var.index != 'CD8']

groups = ['ctr', 'hst', 'ftl']
tissues = CD8_high.obs['tissue'].unique().tolist()
adata_list = []

for group in groups:
    for tissue in tissues:
        subset = CD8_high[(CD8_high.obs['group'] == group) & (CD8_high.obs['tissue'] == tissue)]
        if subset.shape[0] >= 5000:
            sampled_subset = subset[np.random.choice(subset.shape[0], 5000, replace=False)]
        else:
            sampled_subset = subset  
        
        sampled_subset.obs['group'] = f"{group}"
        adata_list.append(sampled_subset.copy())

CD8_high = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
sc.settings.verbosity = 3

In [ ]:
plt.rcParams.update({'font.size': 10})  

In [ ]:
sc.tl.pca(CD4_low, svd_solver="arpack")
sc.pl.pca_variance_ratio(CD4_low, log=False)

In [ ]:
sc.tl.pca(CD4_high, svd_solver="arpack")
sc.pl.pca_variance_ratio(CD4_high, log=False)

In [ ]:
sc.pl.pca_loadings(CD4_low, components = '1,2')
sc.pl.pca_loadings(CD4_high, components = '1,2')

In [ ]:
bom_samples = bom_session.get_sample_ids()
lng_samples = lng_session.get_sample_ids()
lln_samples = lln_session.get_sample_ids()
mln_samples = mln_session.get_sample_ids()
spl_samples = spl_session.get_sample_ids()

In [ ]:
sample_list = bom_samples + lng_samples + lln_samples + mln_samples + spl_samples
sample_list = list(set(sample_list))

In [ ]:
# from preprocessing import pca_df
# grouped_pca = pca_df(sample, singular = False, sample_list=sample_list)

In [ ]:
# from plotting_methods import pca_plot
# pca_plot(grouped_pca, tissue_type, sample_name)

In [ ]:
import harmonypy as hm
harmony_out = hm.run_harmony(CD4_low.obsm['X_pca'], CD4_low.obs, 'sample_id', max_iter_harmony=10, theta = 0)
CD4_low.obsm['X_pca_harmony'] = harmony_out.Z_corr
sc.pp.neighbors(CD4_low, use_rep='X_pca_harmony')
sc.tl.umap(CD4_low)

In [ ]:
harmony_out = hm.run_harmony(CD4_high.obsm['X_pca'], CD4_high.obs, 'sample_id', max_iter_harmony=10, theta = 0)
CD4_high.obsm['X_pca_harmony'] = harmony_out.Z_corr
sc.pp.neighbors(CD4_high, use_rep='X_pca_harmony')
sc.tl.umap(CD4_high)

In [ ]:
markers = list(CD4_low.var_names)

In [ ]:
sc.pl.umap(CD4_low, color= ['group'], cmap='turbo', title = '{} {} Groups'.format('All Tissues', 'CD4 Groups (Age < 40)'))

In [ ]:
sc.pl.umap(CD4_high, color= ['group'], cmap='turbo', title = '{} {} Groups'.format('All Tissues', 'CD4 Groups (Age > 40)'))

In [ ]:
sc.tl.leiden(CD4_low, resolution=0.4, flavor='leidenalg')
sc.tl.leiden(CD4_high, resolution=0.4, flavor='leidenalg')

In [ ]:
sc.pl.umap(CD4_low, color= ['leiden'], cmap='turbo', title = '{} {} Clusters'.format('All Tissues', 'CD4 Clusters (Age < 40)'))

In [ ]:
sc.pl.umap(CD4_high, color= ['leiden'], cmap='turbo', title = '{} {} Clusters'.format('All Tissues', 'CD4 Clusters (Age > 40)'))

In [ ]:
sc.pl.umap(CD4_low, color= ['tissue'], cmap='turbo', title = '{} Tissue Groups'.format('CD4 (Age < 40)'))

In [ ]:
sc.pl.umap(CD4_high, color= ['tissue'], cmap='turbo', title = '{} Tissue Groups'.format('CD4 (Age > 40)'))

In [ ]:
plt.rcParams.update({'font.size': 14})

sc.pl.umap(
    CD4_low,
    color=markers,
    cmap='turbo',
    vmin = 0,
    vmax = 5,
)

In [ ]:
plt.rcParams.update({'font.size': 14})

sc.pl.umap(
    CD4_high,
    color=markers,
    cmap='turbo',
    vmin = 0,
    vmax = 5,
)

In [ ]:
sc.tl.dendrogram(CD4_low, groupby= 'leiden')

In [ ]:
sc.tl.dendrogram(CD4_high, groupby= 'leiden')

In [ ]:
sc.pl.dotplot(CD4_low, markers, swap_axes=True, groupby='leiden', title = "{} {} Dotplot".format('All Samples', "CD4 (Age < 40)"), cmap='RdBu_r', dendrogram=True, vcenter = 0, vmin = -4, vmax = 4)

In [ ]:
sc.pl.dotplot(CD4_high, markers, swap_axes=True, groupby='leiden', title = "{} {} Dotplot".format('All Samples', "CD4 (Age > 40)"), cmap='RdBu_r', dendrogram=True, vcenter = 0, vmin = -4, vmax = 4)

In [ ]:
marker_genes = {
    'Tfh': ['CXCR5', 'PD1'],
    'Th1': ['CXCR3'],
    'Th2': ['CRTH2'],
    'Trm': ['CD103', 'CD69'],
    'Treg': ['CD25', 'FOXP3'],
    'Memory': ['CCR7', 'CD45RA']
}

In [ ]:
sc.pl.dotplot(CD4_low, marker_genes, swap_axes=True, groupby='leiden', cmap='RdBu_r', dendrogram=True, vcenter = 0, vmin = -4, vmax = 4)

In [ ]:
sc.pl.dotplot(CD4_high, marker_genes, swap_axes=True, groupby='leiden', cmap='RdBu_r', dendrogram=True, vcenter = 0, vmin = -4, vmax = 4)

In [ ]:
sc.tl.rank_genes_groups(CD4_low, "leiden", method="t-test")

result = CD4_low.uns["rank_genes_groups"]
groups = result["names"].dtype.names

celltype = {'celltype': []}
cluster_to_genes = {}
for group in groups:
    top_genes = result["names"][group][:3]
    cluster_to_genes[group] = f"{':'.join(top_genes)} ({group})"
    
celltype['celltype'] = [cluster_to_genes[leiden] for leiden in CD4_low.obs['leiden']]

In [ ]:
cell_type_series = pd.Series(celltype['celltype'])
unique_values = cell_type_series.unique()
print(unique_values)

In [ ]:
sc.tl.rank_genes_groups(CD4_high, "leiden", method="t-test")

result = CD4_high.uns["rank_genes_groups"]
groups = result["names"].dtype.names

celltype = {'celltype': []}
cluster_to_genes = {}
for group in groups:
    top_genes = result["names"][group][:3]
    cluster_to_genes[group] = f"{':'.join(top_genes)} ({group})"
    
celltype['celltype'] = [cluster_to_genes[leiden] for leiden in CD4_high.obs['leiden']]

In [ ]:
cell_type_series = pd.Series(celltype['celltype'])
unique_values = cell_type_series.unique()
print(unique_values)